<a href="https://colab.research.google.com/github/aashish7542/ai-learning/blob/main/phase-0-foundations/python/Phase_0_3_Exoplanet_ML_Syntax_Cheat_Sheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 0.3 — Python & ML Syntax Reference
### Pandas · NumPy · Matplotlib · Plotly · Scikit-learn

**Dataset:** Exoplanets (`seaborn` planets dataset)  
**Purpose:** A searchable, executable cheat sheet for the **Phase 0.3.6 mini-project**.

**Your study status:** Phase 0.3 sections **0.3.1–0.3.5 completed** · **0.3.6 checkpoint 1/5 in progress**.

> **How to use:** Open in Google Colab. Use **Runtime → Run all** to run the demonstrations, or run a single cell when you need a reminder. Each example is separate from your actual mini-project; you'll still write and explain your own project solution.
>
> **Important:** The examples use a `*_demo` suffix when constructing illustrative training datasets. These aren't your final project dataset or modeling decisions. The original `df` is never overwritten by the cleaning examples.

**Jump to:** [Imports](#1-imports-and-data) · [Selecting data](#2-selecting-and-filtering) · [Inspecting](#3-inspection-and-cleaning) · [Statistics](#4-statistics-and-exploration) · [Plots](#5-visualization) · [Train/test](#6-training-and-testing-syntax) · [Preprocessing](#7-preprocessing-syntax) · [Validation](#8-validation-and-saving) · [Quick lookup](#9-common-syntax-distinctions)

## 1. Imports and data
Import only the packages you need for a particular cell. These are the main libraries used throughout this subphase.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

### Load the reference dataset
`pd.read_csv(path_or_url)` reads a CSV file into a DataFrame. Google Colab can read this public URL directly.

In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/planets.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
display(df.head())

## 2. Selecting and filtering

| Syntax | Meaning |
|---|---|
| `df.copy()` | Make an independent DataFrame copy |
| `df["mass"]` | Select a **Series** (one column) |
| `df[["mass", "distance"]]` | Select a **DataFrame** (multiple columns) |
| `df.loc[row_condition, ["mass", "distance"]]` | Filter rows and select columns |
| `df.drop(columns=["mass"])` | Remove columns in the returned DataFrame |
| `df.dropna(subset=["mass"])` | Remove only rows where `mass` is missing |
| `df.fillna(value)` | Fill missing values (returns a new DataFrame unless assigned) |
| `df.reset_index(drop=True)` | Reset row labels to 0, 1, 2, … |

**Rule:** Prefer keeping `df` unchanged. Assign the result to another variable when exploring or cleaning.

In [ ]:
# Select one or more columns
mass_series = df["mass"]
physical_features = df[["mass", "distance"]].copy()

# Boolean filtering
heavy_planets = df[df["mass"] > 2]
transit_planets = df[df["method"] == "Transit"]
recent_planets = df[df["year"] >= 2010]

# AND uses &, OR uses |; put each condition in parentheses
heavy_and_recent = df[(df["mass"] > 2) & (df["year"] >= 2010)]
heavy_or_recent = df[(df["mass"] > 2) | (df["year"] >= 2010)]

# .loc chooses rows and columns together
selected = df.loc[df["year"] >= 2010, ["mass", "distance", "method"]]
print(selected.shape)

In [ ]:
# Demonstrate non-destructive cleaning syntax; original df is unchanged
without_mass_column = df.drop(columns=["mass"])
known_masses = df.dropna(subset=["mass"])
filled_example = df["mass"].fillna(df["mass"].median())  # demonstration only
reset_example = known_masses.reset_index(drop=True)

print("Original rows:", len(df), "Known-mass rows:", len(known_masses))

## 3. Inspection and cleaning

| Syntax | Meaning |
|---|---|
| `df.head(5)` / `df.tail(5)` | First / last five rows |
| `df.shape` | Tuple `(rows, columns)`; **no parentheses** |
| `df.columns` | Column labels |
| `df.info()` | Data types and non-null counts |
| `df.describe()` | Summary statistics for numerical columns |
| `df.isna().sum()` | Missing count for each column |
| `df.isna().sum().sort_values(ascending=False)` | Missing counts, largest first |
| `df.duplicated().sum()` | Number of duplicate occurrences after first appearances |
| `df[df.duplicated(keep=False)]` | Display **all** occurrences of duplicated rows |
| `df.drop_duplicates()` | Drop repeated identical rows in a new DataFrame |
| `df["method"].value_counts()` | Category counts |
| `df["mass"].nunique()` | Number of unique non-null values |

**Astronomy warning:** Exact duplicate recorded measurements do **not** prove that two records are the same planet. Do not automatically delete suspected duplicates.

In [ ]:
display(df.head(5))
display(df.tail(5))
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.info()
display(df.describe())

In [ ]:
# Missingness and duplicates
missing_by_column = df.isna().sum().sort_values(ascending=False)
display(missing_by_column)
print("Duplicate occurrences:", df.duplicated().sum())
display(df[df.duplicated(keep=False)].head(10))

# Inspect categories and uniqueness
print("Distinct non-null masses:", df["mass"].nunique())
display(df["method"].value_counts().head(10))

# Syntax demo only; do not assume duplicate rows should be dropped
without_identical_rows = df.drop_duplicates()

## 4. Statistics and exploration

| Syntax | Meaning |
|---|---|
| `df["mass"].mean()` | Arithmetic mean |
| `df["mass"].median()` | Median |
| `df["mass"].std()` | Sample standard deviation (Pandas default) |
| `df["mass"].min()` / `.max()` | Minimum / maximum |
| `df["mass"].quantile(0.75)` | 75th percentile |
| `df.corr(numeric_only=True)` | Pearson correlations among numerical columns |
| `df.groupby("method")["mass"].median()` | Median mass within each method |
| `pd.crosstab(df["method"], df["mass"].isna())` | Cross-tabulate method versus missing mass |

**Remember:** A correlation near zero means weak **linear** association, not necessarily independence or absence of a nonlinear relationship.

In [ ]:
print("Mean:", df["mass"].mean())
print("Median:", df["mass"].median())
print("Standard deviation:", df["mass"].std())
print("Min, max:", df["mass"].min(), df["mass"].max())
print("75th percentile:", df["mass"].quantile(0.75))

display(df.corr(numeric_only=True))
display(df.groupby("method")["orbital_period"].median().sort_values())
display(pd.crosstab(df["method"], df["mass"].isna()))

## 5. Visualization
### 5A. Matplotlib — static plots

| Syntax | Use |
|---|---|
| `plt.hist(values, bins=30)` | Histogram |
| `plt.scatter(x, y, alpha=0.5)` | Scatterplot, partly transparent points |
| `plt.xscale("log")` / `plt.yscale("log")` | Logarithmic x/y axis |
| `plt.xlabel("...")` / `plt.ylabel("...")` | Axis labels |
| `plt.title("...")` | Plot title |
| `plt.show()` | Display the plot |

**Scale distinction:** `plt.xscale("log")` changes the *axis*, while `np.log10(values)` transforms the *data*. Only use either logarithmic approach on strictly positive values.

In [ ]:
# Example: log-transformed histogram of known positive masses
mass_positive = df.loc[df["mass"] > 0, "mass"].dropna()
plt.hist(np.log10(mass_positive), bins=30)
plt.xlabel("log10(Mass in Jupiter masses)")
plt.ylabel("Number of planets")
plt.title("Exoplanet mass distribution")
plt.show()

In [ ]:
# Example: scatterplot with logarithmic axes
pairs = df.dropna(subset=["mass", "orbital_period"]).copy()
pairs = pairs[(pairs["mass"] > 0) & (pairs["orbital_period"] > 0)]
plt.scatter(pairs["mass"], pairs["orbital_period"], alpha=0.5)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Mass (Jupiter masses)")
plt.ylabel("Orbital period (days)")
plt.title("Orbital period versus mass")
plt.show()

### 5B. Plotly — interactive plots

| Syntax | Use |
|---|---|
| `px.histogram(df, x="mass", nbins=30)` | Interactive histogram |
| `px.scatter(df, x="mass", y="distance")` | Interactive scatterplot |
| `px.box(df, x="method", y="orbital_period")` | Group comparison / boxplot |
| `px.scatter_matrix(df, dimensions=[...])` | Pairwise scatterplots |
| `color="method"` | Color points by category |
| `log_x=True, log_y=True` | Logarithmic axes |
| `hover_data=["year"]` | Extra values on hover |
| `fig.show()` | Display the interactive figure |

**Caution:** Missing values may exclude observations from visualizations; compare the resulting sample to the original before interpreting scientific patterns.

In [ ]:
# Interactive group-colored scatterplot
fig = px.scatter(
    pairs,
    x="mass", y="orbital_period",
    color="method",
    log_x=True, log_y=True,
    hover_data=["year", "distance"],
    opacity=0.6,
    title="Exoplanets by discovery method"
)
fig.show()

In [ ]:
# Other plot types — run any line to explore
fig_hist = px.histogram(df, x="mass", nbins=30, title="Mass histogram")
fig_box = px.box(
    df[df["method"].isin(["Transit", "Radial Velocity"])],
    x="method", y="orbital_period", log_y=True,
    title="Orbital period by method"
)
fig_matrix = px.scatter_matrix(
    df, dimensions=["mass", "distance", "orbital_period"],
    title="Numerical feature pairs"
)

fig_box.show()  # replace with fig_hist.show() or fig_matrix.show()

## 6. Training and testing syntax

These examples use **`year` only as a demonstration target**, **not** as a recommendation for your actual mini-project. Define your project's real target and features yourself.

| Syntax | Meaning |
|---|---|
| `X = data[["feature1", "feature2"]]` | Feature matrix |
| `y = data["target"]` | Target Series |
| `train_test_split(X, y, test_size=0.2, random_state=42)` | Random 80/20 split of corresponding feature–target rows |

**Leakage rule:** Split **before** fitting preprocessing that learns parameters (means, category vocabularies, etc.).

In [ ]:
# ILLUSTRATION ONLY — does not finish checkpoint 1 of the mini-project
X_demo = df[["mass", "distance", "method"]].copy()
y_demo = df["year"].copy()

X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=42
)
print("Train:", X_train_demo.shape, y_train_demo.shape)
print("Test:", X_test_demo.shape, y_test_demo.shape)

## 7. Preprocessing syntax

### 7A. Missing-value imputation

| Strategy | Replaces missing values with |
|---|---|
| `"mean"` | Training-set mean |
| `"median"` | Training-set median |
| `"most_frequent"` | Most common training value |
| `"constant"` | Chosen fixed value (`fill_value=...`) |

Do **not** automatically impute scientifically meaningful missing values. This cell demonstrates syntax only.

In [ ]:
# Numerical columns only; fit on train, transform test
numeric_cols = ["mass", "distance"]
num_imputer = SimpleImputer(strategy="median")

numeric_train_filled = num_imputer.fit_transform(X_train_demo[numeric_cols])
numeric_test_filled = num_imputer.transform(X_test_demo[numeric_cols])
print("After numeric imputation:", numeric_train_filled.shape)

### 7B. Standardization

`StandardScaler()` learns training-feature means and standard deviations. After transforming, units are expressed as relative deviations; scaling does **not** establish equal physical importance.

- `.fit(data)` learns statistics.
- `.transform(data)` applies learned statistics.
- `.fit_transform(data)` does both.

**Always fit using training data only.**

In [ ]:
scaler = StandardScaler()
numeric_train_scaled = scaler.fit_transform(numeric_train_filled)
numeric_test_scaled = scaler.transform(numeric_test_filled)

print("Train means after scaling:", numeric_train_scaled.mean(axis=0).round(6))
print("Test means after scaling:", numeric_test_scaled.mean(axis=0).round(6))

### 7C. Categorical encoding

`OneHotEncoder(handle_unknown="ignore")` turns category strings into indicator columns and safely handles previously unseen categories during transformation.

```text
Method           Transit  Radial Velocity
Transit             1            0
Radial Velocity     0            1
```

The actual number of encoded columns depends on the categories learned in the training data.

In [ ]:
cat_encoder = OneHotEncoder(handle_unknown="ignore")
method_train_encoded = cat_encoder.fit_transform(X_train_demo[["method"]])
method_test_encoded = cat_encoder.transform(X_test_demo[["method"]])

print("Learned methods:", cat_encoder.categories_[0])
print("Encoded train shape:", method_train_encoded.shape)

### 7D. Pipelines and `ColumnTransformer`

- `Pipeline([(name, transformer), ...])` runs transformations in order.
- `ColumnTransformer([(name, transformer, column_names), ...])` applies different transformations to different columns.
- Keep preprocessing steps inside a fitted pipeline to reduce leakage and make the workflow reproducible.

This demonstrates preprocessing only; it does **not** train a predictive model.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_demo = ColumnTransformer([
    ("numeric", numeric_pipeline, ["mass", "distance"]),
    ("categorical", categorical_pipeline, ["method"])
])

prepared_train_demo = preprocessor_demo.fit_transform(X_train_demo)
prepared_test_demo = preprocessor_demo.transform(X_test_demo)
print("Processed train shape:", prepared_train_demo.shape)
print("Processed test shape:", prepared_test_demo.shape)

## 8. Validation and saving

| Syntax | Meaning |
|---|---|
| `len(df)` | Count rows |
| `df.shape[0]` / `df.shape[1]` | Rows / columns |
| `df.isna().any()` | Boolean for each column: is anything missing? |
| `df.isna().sum().sum()` | Total missing entries across columns |
| `assert condition` | Raise `AssertionError` if a condition is false |
| `df.to_csv("file.csv", index=False)` | Write CSV without saving the index |

**Tip:** `assert` is a development sanity check, not user-input validation or a full testing framework.

In [ ]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Any missing by column:")
display(df.isna().any())
print("Total missing cells:", df.isna().sum().sum())

assert len(X_demo) == len(y_demo)
assert len(X_train_demo) == len(y_train_demo)
assert len(X_test_demo) == len(y_test_demo)
print("Alignment checks passed.")

In [ ]:
# Optional: save a COPY to the Colab runtime (not your finished project)
# Uncomment to run:
# df.to_csv("exoplanets_reference_copy.csv", index=False)

## 9. Common syntax distinctions

| Syntax | Interpret it as |
|---|---|
| `df.shape` | Attribute — no `()` |
| `df.head()` | Method call — needs `()` |
| `df["mass"]` | One-column **Series** |
| `df[["mass"]]` | One-column **DataFrame** |
| `df == 5` | Elementwise comparison across a DataFrame |
| `df["mass"] == 5` | Elementwise comparison within one column |
| `df.isna()` | Find missing entries; returns booleans |
| `df.dropna()` | Exclude rows with **any** missing entry (default) |
| `fit()` | Learn from data |
| `transform()` | Apply what was learned |
| `fit_transform()` | Learn and apply on the same data |
| `df[condition]` | Filter rows by a Boolean condition |
| `df.loc[condition, columns]` | Filter rows **and** specify columns |

## 10. Mini-project reminder — your turn

**0.3.6 / Checkpoint 1 of 5: Define the dataset**

From the original `df`, write code **yourself** to:

1. Make a **separate copy** with `mass`, `distance`, `method`, and `orbital_period`.
2. Remove rows **only** when the target `orbital_period` is missing.
3. Split the resulting DataFrame into feature matrix `X` and target `y`.
4. Print the shapes of both.

Avoid overwriting `df`. This reference deliberately does **not** give your solution.

**After you submit your code:** We'll review correctness, coding habits, and what remains to complete the five mini-project checkpoints.

---
**Reference conventions:** Real astronomical observations, but demonstration preprocessing choices are not scientific recommendations. Exoplanet `distance` means distance from Earth to the system, not orbital radius. This historic dataset has limited feature coverage and strong observational selection effects.